# 3. Train classical baselines — Member 3

Run notebook 02 first. Running this notebook top to bottom trains the models directly and saves fold scores, configurations and held-out predictions. No baseline script command or uncommenting is required.

Dummy, LR, linear SVM, RF and XGBoost use the same 58 time features and all 22 LOSO folds. This is the expensive milestone notebook. Re-running a training cell retrains that model and replaces its saved outputs; running all cells retrains all models.

Linear SVM keeps full LOSO tractable. RBF SVM and tuning are deferred. Primary macro F1 averages all 12 labels, assigning zero for absent labels; a separate present-class metric makes varying coverage visible.

In [ ]:
from pathlib import Path
import os
import sys
# Find the project from the workspace, project folder, or notebooks folder.
ROOT = next((p for p in [Path.cwd(), Path.cwd() / 'harth_project', Path.cwd().parent]
             if (p / 'har_utils.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from the IMLProj workspace or harth_project folder.')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('OMP_NUM_THREADS', '1')
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from har_utils import *

In [ ]:
X, meta = load_windows(2)
X_time = X[feature_columns(X)]
record_environment()
baseline_models = models()
display(pd.Series(baseline_models).astype(str).to_frame('model'))
print('Training windows:', len(meta), '| features:', X_time.shape[1])

## Train each fixed baseline

Each cell calls the same evaluate() helper, which verifies subject-disjoint splits and fits preprocessing using training participants only. If interrupted between models, resume with the next unfinished model rather than running all cells again.

In [ ]:
dummy_scores, _ = evaluate(X_time, meta, baseline_models['Dummy'], 'Dummy', 'loso')
save_summary()

In [ ]:
lr_scores, _ = evaluate(X_time, meta, baseline_models['LR'], 'LR', 'loso')
save_summary()

In [ ]:
svm_scores, _ = evaluate(X_time, meta, baseline_models['LinearSVM'], 'LinearSVM', 'loso')
save_summary()

In [ ]:
rf_scores, _ = evaluate(X_time, meta, baseline_models['RF'], 'RF', 'loso')
save_summary()

In [ ]:
xgb_scores, _ = evaluate(X_time, meta, baseline_models['XGBoost'], 'XGBoost', 'loso')
save_summary()

## Compare the completed models

These cells load saved outputs, so you can revisit them without re-running the training cells above.

In [ ]:
scores = pd.concat([pd.read_csv(ROOT / 'results' / f'{name}_loso_folds.csv')
                    for name in baseline_models])
display(summarize(scores).round(4))
pivot = scores.pivot(index='test_subjects', columns='experiment', values='macro_f1')
display(pivot.round(3))
pivot.plot.box(figsize=(9, 5), title='Between-subject variation')
plt.ylabel('12-class macro F1'); plt.tight_layout(); plt.show()

## Optimistic random-window diagnostic

This cell also trains RF. Participants and nearby recording regions are shared across the random split; it does not estimate unseen-person performance. Differences from LOSO also reflect split composition and training size.

In [ ]:
random_scores, _ = evaluate(X_time, meta, baseline_models['RF'], 'RF', 'random')
save_summary()
display(summarize(pd.concat([scores[scores.experiment == 'RF'], random_scores])).round(4))

## Handoff to Member 4

Notebook 04 uses results/RF_loso_folds.csv and results/RF_loso_predictions.csv. Share these files and their configuration, or keep using the same project folder. Notebook 04 performs error analysis without fitting more models. Stop there for the milestone.